# 00 – Download data

**Formål:** Hente månedlige **registrerede boligindbrud pr. kvarter** for hele Holland fra politiets/CBS' open data og gemme dem i `data/raw/`.

| | |
|---|---|
| **Kilde** | Politie / CBS, tabel **47022NED** – *Geregistreerde misdrijven; soort misdrijf; wijk, buurt; maandcijfers* (licens CC-BY 4.0) |
| **Periode** | Januar 2012 → seneste offentliggjorte måned |
| **Output** | `data/raw/47022NED_burglary_monthly_buurt.csv.gz` |
| **Køretid** | Amsterdam-test: få sekunder · Hele Holland: ca. 5 minutter (hos os) |

> Denne notebook skal kun køres **én gang**. Alle senere notebooks indlæser den gemte fil.

**Den anden datakilde**, CBS' kvarter-nøgletal (`kwb2024.xlsx`), er downloadet manuelt fra CBS' hjemmeside og ligger allerede i `data/raw/`.

## Hvorfor henter vi data med kode (API) i stedet for at klikke filen ned?

Et **API** (*Application Programming Interface*) er en måde for to computere at tale sammen. Det svarer til en tjener på en restaurant (eksemplet fra Lecture 2): vi bestiller ("giv mig boligindbrud i alle kvarterer"), tjeneren henter det i køkkenet (CBS' database) og kommer tilbage med data.

Vi bruger API'et af to grunde:
1. **Datasættet er for stort** til hjemmesidens download-knap (2.607.210 rækker).
2. **Det kan gentages (reproducerbarhed):** alle, der kører denne notebook, får præcis de samme data. Det kan vi dokumentere i rapportens metodeafsnit.

Det er den samme idé som API-opgaverne med World Bank-data i Exercise 2.

## Trin 1 – Installer pakker

`%pip install` installerer Python-pakker i **det miljø (kernel), notebooken bruger**. `-r ../requirements.txt` betyder "installér alt, der står i filen `requirements.txt`" (`../` = én mappe op, fordi notebooken ligger i `notebooks/`).

> Kør kun cellen, første gang du bruger et nyt miljø. Klik bagefter **Restart** i værktøjslinjen, så Python kan se de nye pakker.

In [ ]:
%pip install -r ../requirements.txt requests

## Trin 2 – Importér pakker

| Pakke | Bruges til |
|---|---|
| `requests` | At sende forespørgsler til en hjemmeside/API og modtage svaret (som når en browser henter en side) |
| `pandas` | At arbejde med tabeller (DataFrames), ligesom i alle øvelserne |
| `os` | At tjekke, om en fil allerede findes på computeren |

`import pandas as pd` betyder, at vi kan skrive `pd` i stedet for `pandas` – det er bare en forkortelse, alle bruger.

In [ ]:
import os                # check whether a file already exists
import requests          # download data from the internet
import pandas as pd      # work with tables (DataFrames)

## Trin 3 – Adressen på tabellen

`URL` er adressen på tabellen hos CBS. Den består af flere dele:

| Del | Betydning |
|---|---|
| `https://dataderden.cbs.nl` | CBS' server til data fra **andre** organisationer (her politiet). "Derden" = tredjeparter |
| `/ODataFeed/OData` | Den type adgang, vi bruger. **OData** er et standardformat til tabeller over internettet. Vi skal bruge **ODataFeed** og ikke ODataApi, fordi kun ODataFeed tillader at hente data i flere bidder (se trin 4) |
| `/47022NED` | Tabellens ID (månedstal pr. kvarter) |
| `/TypedDataSet` | Selve data, hvor koderne står som koder (fx `BU0363AA01`) og tallene som tal |

Tabellen har 4 kolonner, vi bruger, + et løbenummer:

| Kolonne | Betydning | Eksempel |
|---|---|---|
| `SoortMisdrijf` | Forbrydelsestype | `1.1.1 ` = boligindbrud (*Diefstal/inbraak woning*) |
| `WijkenEnBuurten` | Område-kode | `BU0363AA01` (BU = buurt/kvarter, 0363 = Amsterdam) |
| `Perioden` | Måned | `2012MM01` = januar 2012 |
| `GeregistreerdeMisdrijven_1` | Antal registrerede forbrydelser | 0, 1, 2 … |
| `ID` | Løbenummer fra CBS | Bruges ikke |

Navne skrevet med STORE BOGSTAVER (`URL`, `PAGE_SIZE`) er en konvention for **konstanter**: værdier, der ikke ændrer sig, mens koden kører.

In [ ]:
# Endpoint of the police/CBS table with monthly registered crimes per neighbourhood (47022NED)
URL = "https://dataderden.cbs.nl/ODataFeed/OData/47022NED/TypedDataSet"

# The server returns at most 10,000 rows per request, so we download in pages of this size
PAGE_SIZE = 10000

# Output file for the final dataset (.gz = compressed, roughly 10 times smaller)
OUTPUT_FILE = "../data/raw/47022NED_burglary_monthly_buurt.csv.gz"

## Trin 4 – Funktionen, der henter data

### Hvad er en funktion?
En **funktion** er et stykke kode med et navn, som man kan genbruge. Vi skriver koden én gang (`def download(...)`) og kan så kalde den flere gange med forskellige input, fx først for Amsterdam og bagefter for hele Holland. Det kender I fra `test_prime` i Exercise 1.

- **Input (parameter):** `filter_text` – en tekst, der fortæller hvilke rækker vi vil have.
- **Output (return):** en pandas-tabel (DataFrame) med alle rækkerne.

### Hvorfor en løkke? (pagination)
Serveren sender **højst 10.000 rækker ad gangen**. Hele Holland er 2.607.210 rækker, så vi skal spørge 261 gange. Det hedder **pagination** (at hente "side for side"), ligesom når en webshop viser 20 varer pr. side.

Vi styrer det med to parametre, som vi sender med til serveren:
- `$top=10000` → "giv mig højst 10.000 rækker"
- `$skip=20000` → "spring de første 20.000 over" (dem har vi allerede)

```
Forespørgsel 1: skip=0       → række 1–10.000
Forespørgsel 2: skip=10000   → række 10.001–20.000
Forespørgsel 3: skip=20000   → række 20.001–30.000
...
Sidste:         tom side      → stop
```

### De fire parametre, vi sender
| Parameter | Betydning |
|---|---|
| `$filter` | Hvilke rækker vi vil have (se trin 5) |
| `$format=json` | Svaret skal være i **JSON**-format (tekst, der ligner Python-dictionaries – samme format som i Exercise 2) |
| `$top` | Max antal rækker pr. side |
| `$skip` | Antal rækker, der skal springes over |

`$`-tegnet foran navnene er en del af OData-standarden.

In [ ]:
def download(filter_text):
    """
    Download all rows from the CBS table that match filter_text and return them as a DataFrame.

    The server returns at most PAGE_SIZE rows per request, so the function downloads page by page
    (pagination) until the server returns an empty page.
    """
    rows = []   # empty list that collects the rows from all pages
    skip = 0    # number of rows already downloaded (starts at 0)

    while True:   # keep going until we stop the loop with "break"

        # 1) Build the request: which rows, which format, and which "page"
        params = {
            "$filter": filter_text,   # which rows (e.g. only residential burglary in Amsterdam)
            "$format": "json",        # return the response as JSON
            "$top": PAGE_SIZE,        # max 10,000 rows per page
            "$skip": skip,            # skip the rows we already have
        }

        # 2) Send the request to the server. timeout=120: give up if the server does not respond within 120 seconds
        response = requests.get(URL, params=params, timeout=120)

        # 3) Stop with a clear error message if the server returns an error (e.g. 500 Server Error).
        #    Without this line the code would continue and fail later with a less clear message
        response.raise_for_status()

        # 4) Convert the JSON response to Python. The rows are stored under the key "value" (a list of dictionaries)
        page = response.json()["value"]

        # 5) An empty page means there are no more rows -> stop the loop
        if not page:
            break

        # 6) Add the rows from this page to the collected list and move 10,000 rows ahead to the next page
        rows += page
        skip += PAGE_SIZE

        # 7) Show progress so we can see that the code is working (":," adds a thousands separator, e.g. 20,000)
        print(f"{len(rows):,} rækker hentet")

    # 8) Convert the list of rows to a table (DataFrame) and return it to the caller
    return pd.DataFrame(rows)

> **Bemærk:** Cellen ovenfor viser **intet output**, når den køres. Den *definerer* bare funktionen. Koden inde i funktionen køres først, når vi *kalder* den i trin 6.
>
> Efter en **Restart** glemmer Python alt, så cellerne i trin 2–4 skal køres igen, før funktionen kan bruges.

## Trin 5 – Filteret: hvilke rækker vil vi have?

Filteret er skrevet i **OData-sprog**, som serveren forstår:

| Del af filteret | Betydning |
|---|---|
| `SoortMisdrijf eq '1.1.1 '` | Forbrydelsestype **lig med** (`eq` = *equals*) `1.1.1 ` = boligindbrud |
| `and` | Begge betingelser skal være opfyldt |
| `substringof('BU0363', WijkenEnBuurten)` | Område-koden **indeholder** teksten `BU0363` = alle kvarterer i Amsterdam |
| `substringof('BU', WijkenEnBuurten)` | Område-koden indeholder `BU` = **alle kvarterer i hele landet** |

**Faldgruber, vi selv er faldet i:**
- Der er et **mellemrum** efter `1.1.1` i CBS' data (`'1.1.1 '`). Uden mellemrummet matcher intet.
- Store og små bogstaver tæller: `SoortMisdrijf`, ikke `soortMisdrijf`.
- Tekstværdier skal stå i **enkelte anførselstegn**: `'BU0363'`.

**Hvorfor kun `BU` (kvarterer)?** Tabellen indeholder også hele kommuner (`GM…`), bydele (`WK…`) og landet (`NL…`). Vi vil kun have det mindste niveau, kvarterer, fordi det er dem, vi forudsiger for.

**Hvorfor kun boligindbrud?** Det er vores target. Boligindbrud anmeldes næsten altid (forsikringen kræver en politianmeldelse), så registrerede indbrud ligger tæt på det faktiske antal. Det mindsker *label bias* (Lecture 2 og 8).

## Trin 6 – Test på Amsterdam

### Hvorfor teste først?
Det er som at **smage på suppen, før man laver en hel gryde**. Hele Holland tager flere minutter at hente. Hvis vi har lavet en stavefejl i filteret eller brugt den forkerte adresse, vil vi hellere opdage det efter 5 sekunder end efter 5 minutter. Amsterdam er stor nok til at være en rigtig test, men lille nok til at gå hurtigt.

### Hvad gør hver linje i cellen nedenfor?
| Kode | Hvad sker der | Med andre ord |
|---|---|---|
| `test = download("...")` | Kører funktionen fra trin 4 med Amsterdam-filteret og gemmer resultatet i variablen `test` | "Hent Amsterdam, og læg tabellen i en kasse, der hedder `test`" |
| `print(test.shape)` | Viser tabellens størrelse som (rækker, kolonner) | "Hvor stor er tabellen?" |
| `test.head()` | Viser de første 5 rækker | "Lad mig kigge på starten af tabellen" |

**Hvorfor står der `print()` om den ene, men ikke den anden?** Jupyter viser automatisk resultatet af den **sidste** linje i en celle. Alle andre linjer skal have `print()` for at blive vist. Derfor står `test.head()` sidst.

### Sådan læser du outputtet
- Først kommer `10,000 rækker hentet`, `20,000 rækker hentet` … Det er funktionens fremskridt: én linje pr. side.
- `(91509, 5)` betyder **91.509 rækker og 5 kolonner**.
- Tabellen viser fx: `BU0363AA01 | 2012MM01 | 0` = *"I kvarter BU0363AA01 var der 0 registrerede boligindbrud i januar 2012."*

In [ ]:
# Residential burglary ('1.1.1 ') in all Amsterdam neighbourhoods ('BU0363')
test = download("SoortMisdrijf eq '1.1.1 ' and substringof('BU0363', WijkenEnBuurten)")

print(test.shape)   # (number of rows, number of columns)
test.head()         # first 5 rows

### Tjek: giver tallene mening?
At hente data er ikke nok – vi skal også **kontrollere**, at de er rigtige (data quality, Lecture 2). Tænk på det som at **tælle byttepengene**, før man går fra kassen.

Logikken er simpel: hvert kvarter skal have **én række pr. måned**. Så skal dette passe:

> **antal kvarterer × antal måneder = antal rækker**

Passer det, mangler der ingen måneder for noget kvarter, og der er ingen dubletter.

### Hvad gør hver linje i cellen nedenfor?
| Kode | Hvad sker der | Eksempel |
|---|---|---|
| `test["WijkenEnBuurten"].nunique()` | Tæller hvor mange **forskellige** værdier der er i kolonnen (*n unique*). Dubletter tælles kun én gang | Kvarter A står 177 gange, men tæller som 1 → **517** kvarterer |
| `test["Perioden"].nunique()` | Tæller forskellige måneder | **177** måneder |
| `.min()` og `.max()` | Første og sidste måned. Det virker på tekst, fordi formatet `ÅÅÅÅMMmm` sorterer i den rigtige rækkefølge | `2012MM01` → `2026MM09` |
| `n_buurten * n_months` | Det **forventede** antal rækker | 517 × 177 = 91.509 |
| `len(test)` | Det **faktiske** antal rækker | 91.509 ✓ |
| `(test["GeregistreerdeMisdrijven_1"] == 0)` | Spørger hver række: "er antallet 0?" og svarer `True` eller `False` | `[True, True, False, True …]` |
| `.mean()` | Gennemsnittet af `True`/`False`. Python regner `True` = 1 og `False` = 0, så gennemsnittet er **andelen** af rækker med 0 indbrud | 3 ud af 4 er `True` → 0,75 = 75 % |
| `round(…, 3)` | Afrunder til 3 decimaler | 0,7532… → 0,753 |

**Resultat for Amsterdam:** 517 kvarterer × 177 måneder (jan 2012–sep 2026) = **91.509 rækker** ✓

In [ ]:
n_buurten = test["WijkenEnBuurten"].nunique()   # number of distinct neighbourhoods
n_months = test["Perioden"].nunique()           # number of distinct months

print("Kvarterer:", n_buurten)
print("Måneder:", n_months, "| fra", test["Perioden"].min(), "til", test["Perioden"].max())
print("Kvarterer × måneder =", n_buurten * n_months, "| rækker i data =", len(test))
print("Andel måneder med 0 indbrud:", round((test["GeregistreerdeMisdrijven_1"] == 0).mean(), 3))

### Hvorfor er andelen af nuller vigtig?
Den sidste linje viser, hvor stor en andel af kvarter-måneder der har **0 indbrud** – og den er høj. Det kaldes **sparse data** (data med rigtig mange nuller), og underviseren nævnte det i sin feedback.

Forestil dig et kvarter med 3 indbrud om året:

| Måned | jan | feb | mar | apr | maj | jun | jul | aug | sep | okt | nov | dec |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Indbrud | 0 | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 1 |

Hvis modellen skulle gætte **måned for måned**, ville den næsten altid have ret ved at sige "0" – og så har den ikke lært noget. Lægger vi månederne sammen til **sæsonen okt–mar**, får vi i stedet ét tal (her 3), som er langt mere stabilt og meningsfuldt. Det er derfor, vi senere arbejder med **sæsoner** i stedet for måneder.

## Trin 7 – Hent hele Holland og gem filen

Nu henter vi det rigtige datasæt: samme filter som i testen, men `'BU'` i stedet for `'BU0363'` – altså **alle kvarterer i landet**. Hos os tog det ca. 5 minutter.

### Hvorfor tjekker vi først, om filen findes?
Det er som at **kigge i køleskabet, før man går ud og handler**. Har vi allerede hentet data én gang, er der ingen grund til at vente 5 minutter igen. Koden træffer en beslutning:

```
Findes filen allerede?
   ├── JA  → læs den fra computeren (tager sekunder)
   └── NEJ → hent fra CBS (tager minutter) → gem den, så vi har den næste gang
```

### Hvad gør hver linje i cellen nedenfor?
| Kode | Hvad sker der |
|---|---|
| `if os.path.exists(OUTPUT_FILE):` | Spørger computeren: "findes denne fil?" Svaret er `True` eller `False` |
| `burglary = pd.read_csv(OUTPUT_FILE)` | **Hvis ja:** læs filen ind som en tabel. pandas pakker selv `.gz`-filen ud |
| `else:` | **Hvis nej:** gør i stedet det, der står nedenunder |
| `burglary = download("...'BU'...")` | Hent alle kvarterer via funktionen fra trin 4 |
| `burglary.to_csv(OUTPUT_FILE, index=False)` | Gem tabellen som en fil |
| `print(burglary.shape)` og `burglary.head()` | Vis størrelse og de første rækker (som i trin 6) |

### Hvad betyder `index=False`?
pandas giver hver række et nummer (0, 1, 2 …), kaldet **index**. Det er ikke rigtige data, bare pandas' egen nummerering. Uden `index=False` ville det nummer blive gemt som en ekstra, meningsløs kolonne i filen.

### Hvorfor `.csv.gz`?
`.gz` er komprimering, ligesom en zip-fil. Den almindelige CSV-fil ville fylde over 100 MB, som er GitHubs grænse pr. fil. Komprimeret fylder den langt mindre.

### Hvorfor gemmer vi data uændret?
Filer i `data/raw/` er data, **præcis som de blev hentet**. Al oprydning sker i de næste notebooks. Så kan vi altid gå tilbage til originalen, hvis vi laver en fejl senere.

In [ ]:
if os.path.exists(OUTPUT_FILE):
    # File already exists: load it instead of downloading again
    burglary = pd.read_csv(OUTPUT_FILE)
    print("Fil fundet – indlæst fra", OUTPUT_FILE)
else:
    # File does not exist: download residential burglary ('1.1.1 ') for all neighbourhoods in the country ('BU')
    burglary = download("SoortMisdrijf eq '1.1.1 ' and substringof('BU', WijkenEnBuurten)")
    burglary.to_csv(OUTPUT_FILE, index=False)   # index=False: do not save the pandas row numbers as an extra column
    print("Downloadet og gemt i", OUTPUT_FILE)

print(burglary.shape)
burglary.head()

### Hvad vi fik
- **Første gang** cellen blev kørt, viste den `10,000 rækker hentet` … `2,607,210 rækker hentet` og derefter `Downloaded and saved to …`.
- **Anden gang** viste den `File found - loaded from …` på få sekunder. Det beviser, at fil-tjekket virker.
- `(2607210, 5)` = **2.607.210 rækker og 5 kolonner**. Første række er fx `BU16800000 | 2012MM01 | 1` = *"1 registreret boligindbrud i kvarter BU16800000 i januar 2012."*

## Trin 8 – Tjek det fulde datasæt

Samme kontrol som for Amsterdam, nu for hele landet – plus to ekstra tjek.

| Kode | Hvad sker der |
|---|---|
| `nunique()`, `min()`, `max()`, `len()` | Samme som i trin 6: antal kvarterer, måneder, første/sidste måned og antal rækker |
| `burglary.isna()` | Spørger hver celle i tabellen: "er du tom?" (`True`/`False`). *NA* = *Not Available* = manglende værdi |
| `.sum()` | Lægger `True`'erne sammen **pr. kolonne** = antal tomme celler i hver kolonne. Vi håber på 0 overalt |
| `burglary["GeregistreerdeMisdrijven_1"].sum()` | Lægger alle indbrud sammen = det samlede antal registrerede boligindbrud |
| `int(…)` | Gør tallet til et helt tal, så det vises pænt (uden `.0`) |

In [ ]:
n_buurten = burglary["WijkenEnBuurten"].nunique()
n_months = burglary["Perioden"].nunique()

print("Kvarterer:", n_buurten)
print("Måneder:", n_months, "| fra", burglary["Perioden"].min(), "til", burglary["Perioden"].max())
print("Kvarterer × måneder =", n_buurten * n_months, "| rækker i data =", len(burglary))
print("Manglende værdier pr. kolonne:")
print(burglary.isna().sum())
print("Samlet antal registrerede boligindbrud:", int(burglary["GeregistreerdeMisdrijven_1"].sum()))

### Resultat og hvad det betyder

| | |
|---|---|
| **Kvarterer** | 14.730 |
| **Måneder** | 177 (januar 2012 – september 2026) |
| **Rækker** | 2.607.210 = 14.730 × 177 ✓ |
| **Manglende værdier** | 0 i alle kolonner ✓ |
| **Boligindbrud i alt** | 661.753 |

**Sådan tolker vi tallene:**
- **Data er komplette.** Hvert kvarter har præcis én række pr. måned, og ingen celler er tomme. Vi skal altså ikke bruge tid på at lappe huller.
- 661.753 indbrud over 177 måneder er ca. **45.000 boligindbrud om året** i hele Holland.
- Men fordelt på næsten 15.000 kvarterer er det kun **ca. 3 indbrud pr. kvarter pr. år** – eller ca. 0,25 om måneden. Det bekræfter det, vi så i Amsterdam: **langt de fleste kvarter-måneder har 0 indbrud** (sparse data). Derfor skal vi:
  1. lægge månederne sammen til **sæsoner (okt–mar)**, og
  2. frasortere **meget små kvarterer**, hvor tallene mest er tilfældigheder.

## Opsummering

| | |
|---|---|
| **Hvad vi har** | Antal registrerede boligindbrud pr. kvarter pr. måned for hele Holland: 14.730 kvarterer × 177 måneder = 2.607.210 rækker, uden manglende værdier |
| **Gemt i** | `data/raw/47022NED_burglary_monthly_buurt.csv.gz` |
| **Vigtigste observation** | Ca. 3 indbrud pr. kvarter pr. år → mange nuller → vi skal arbejde med sæsoner og frasortere små kvarterer |
| **Næste notebook** | `01_explore_data.ipynb`: hvordan fordeler indbrud sig over året og mellem kvarterer? Hvor mange kvarterer er for små? Hvilke kolonner fra `kwb2024.xlsx` skal vi bruge? |

### Mulige eksamensspørgsmål til denne notebook
- *Hvorfor henter I data via API?* → Datasættet er for stort til manuel download (2,6 mio. rækker), og det gør dataindsamlingen reproducerbar.
- *Hvad gør `while`-løkken?* → Pagination: serveren giver max 10.000 rækker pr. gang, så vi henter side for side (261 gange), indtil en side er tom.
- *Hvorfor `raise_for_status()`?* → Så koden stopper med en tydelig fejl, hvis serveren svarer med en fejl, i stedet for at fortsætte med forkerte eller tomme data.
- *Hvorfor tjekker I, om filen findes?* → Så vi ikke downloader igen hver gang, notebooken køres. Det sparer tid og belaster ikke CBS' server unødigt.
- *Hvorfor kun boligindbrud?* → Det anmeldes næsten altid pga. forsikring, så registrerede tal er tæt på de faktiske. Det mindsker label bias.
- *Hvorfor kun kvarter-niveau (BU)?* → Det er den enhed, kommunen prioriterer, og det giver flest observationer.
- *Hvordan ved I, at data er komplette?* → 14.730 kvarterer × 177 måneder = 2.607.210 rækker, præcis som i data, og 0 manglende værdier.
- *Hvad betyder `(x == 0).mean()`?* → Andelen af rækker med 0 indbrud, fordi `True` tæller som 1 og `False` som 0.
- *Hvorfor arbejder I med sæsoner og ikke måneder?* → Ca. 3 indbrud pr. kvarter pr. år giver næsten kun nuller pr. måned (sparse data). Sæsoner giver mere stabile tal og passer til kommunens beslutning før vinteren.